In [1]:
import xarray as xr

In [3]:
ds = xr.open_dataset('../../NHCS/hincast_1980-2015/croco_avg_Y1980M01.nc')
ds.temp

<xarray.DataArray 'temp' (time: 1, s_rho: 32, eta_rho: 542, xi_rho: 602)> Size: 42MB
[10441088 values with dtype=float32]
Coordinates:
  * xi_rho   (xi_rho) float64 5kB 1.0 2.0 3.0 4.0 ... 599.0 600.0 601.0 602.0
  * eta_rho  (eta_rho) float64 4kB 1.0 2.0 3.0 4.0 ... 539.0 540.0 541.0 542.0
  * s_rho    (s_rho) float64 256B -0.9844 -0.9531 -0.9219 ... -0.04688 -0.01562
    lon_rho  (eta_rho, xi_rho) float64 3MB ...
    lat_rho  (eta_rho, xi_rho) float64 3MB ...
  * time     (time) float32 4B 1.339e+06
Attributes:
    long_name:      averaged potential temperature
    units:          Celsius
    field:          temperature, scalar, series
    standard_name:  sea_water_potential_temperature

In [5]:
from pathlib import Path
import re
import numpy as np
import xarray as xr

# --- Files and times ---
files = sorted(Path("../../NHCS/hincast_1980-2015/").glob("croco_avg_Y*M*.nc"))

def extract_datetime_from_filename(path):
    match = re.search(r"Y(\d{4})M(\d{2})", path.name)
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        return np.datetime64(f"{year}-{month:02d}")
    else:
        raise ValueError(f"Fecha no encontrada en nombre: {path.name}")

time_values = [extract_datetime_from_filename(f) for f in files]


In [6]:
def load_temp_depths(path, time_val, depths):
    time_val = np.datetime64(time_val, 'ns')
    ds = xr.open_dataset(path, chunks={"time": 1})

    # --- Depth computation ---
    h      = ds['h'].values
    hc     = float(ds['hc'].values)
    Cs_r   = ds['Cs_r'].values
    s_rho  = ds['s_rho'].values
    vtrans = int(ds['Vtransform'].values)

    if vtrans == 1:
        z = (hc * s_rho[:, None, None]
             + (h[None, :, :] - hc) * Cs_r[:, None, None])
    elif vtrans == 2:
        z = ((hc * s_rho[:, None, None] + h[None, :, :] * Cs_r[:, None, None])
             / (hc + h[None, :, :])
             * h[None, :, :])

    temp = ds['temp'].isel(time=0).values   # (s_rho, eta_rho, xi_rho)
    ny, nx = h.shape
    jj, ii = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')

    # --- Extract all depths into a 3D array (depth, eta_rho, xi_rho) ---
    stack = np.full((len(depths), ny, nx), np.nan, dtype=np.float32)

    for k, d in enumerate(depths):
        idx = np.argmin(np.abs(z - (-float(d))), axis=0)
        field = temp[idx, jj, ii].astype(np.float32)
        field[h < d] = np.nan
        stack[k] = field

    # --- Wrap as DataArray with depth dim ---
    lon_1d = ds['lon_rho'].isel(eta_rho=0).values
    lat_1d = ds['lat_rho'].isel(xi_rho=0).values

    temp_da = xr.DataArray(
        stack,
        dims=['depth', 'lat', 'lon'],
        coords={
            'depth': depths,
            'lat':   lat_1d,
            'lon':   lon_1d,
        },
        attrs={'long_name': 'Temperature', 'units': 'Celsius'}
    )

    return temp_da.expand_dims(time=[time_val])


# --- Run ---
depths = [5] + list(range(10, 101, 10)) + list(range(120, 301, 20))

from tqdm.notebook import tqdm

temp_list = [load_temp_depths(f, t, depths) for f, t in tqdm(zip(files, time_values), total=len(files), desc="Loading temp depths")]
temp_all = xr.concat(temp_list, dim="time")

print(temp_all)

Loading temp depths:   0%|          | 0/432 [00:00<?, ?it/s]

<xarray.DataArray (time: 432, depth: 21, lat: 542, lon: 602)> Size: 12GB
array([[[[20.187956 , 20.176714 , 20.24205  , ...,  0.       ,
           0.       ,  0.       ],
         [20.1992   , 20.289068 , 20.375937 , ...,  0.       ,
           0.       ,  0.       ],
         [20.23746  , 20.348139 , 20.458187 , ...,  0.       ,
           0.       ,  0.       ],
         ...,
         [26.64544  , 26.665993 , 26.61425  , ...,  0.       ,
           0.       ,  0.       ],
         [26.64165  , 26.67271  , 26.637682 , ...,  0.       ,
           0.       ,  0.       ],
         [26.64862  , 26.65559  , 26.628126 , ...,  0.       ,
           0.       ,  0.       ]],

        [[19.424913 , 19.407413 , 19.442787 , ...,  0.       ,
           0.       ,  0.       ],
         [19.442415 , 19.588814 , 19.673569 , ...,  0.       ,
           0.       ,  0.       ],
         [19.506538 , 19.682528 , 19.787685 , ...,  0.       ,
           0.       ,  0.       ],
...
         [10.85611  , 10.

In [18]:
# temp_all.to_dataset(name='temp').to_netcdf('Temp.nc')